# Large Scale vs Small Scale: Is the THC Driven by Large-Scale Density Differences?

**Author:** Halldór Björnsson  
**Companion Interactive Notebook to Lecture:** `MMCI.pdf`

---

## Overview
In the early 1960s, Henry Stommel published two landmark theories on the ocean overturning circulation:
1. **The Stommel–Arons Model (1960):** A dynamical, geostrophic planetary model of abyssal flow on a rotating sphere.
2. **The Thermohaline Box Model (1961):** A conceptual two-box model assuming the circulation is driven by large-scale density differences ($q = k \Delta\rho$).

This notebook explores the mathematical foundations of both theories, highlights the **Straub (1996) inconsistency** between them, examines the role of **Southern Ocean winds (Toggweiler & Samuels 1995; Gnanadesikan 1999)**, and demonstrates through numerical simulations how **Southern Ocean winds generate an apparent linear relation between $q$ and $\Delta\rho$ that is an emergent consequence rather than a causal driver** (Greatbatch & Lu 2003).


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Import the model modules
from stommel_box import StommelBoxModel
from stommel_arons import StommelAronsModel
from gnanadesikan_pycnocline import GnanadesikanPycnoclineModel
from gnanadesikan_f_vs_overturning import plot_single_f_vs_overturning
from wind_experiment import run_wind_experiment


---
## 1. The Stommel (1961) Two-Box Model & Advective Instability

Stommel (1961) considered two well-mixed boxes:
- **Equatorial Box (Box 1):** Warm and saline ($T_1, S_1$).
- **Polar Box (Box 2):** Cold and fresh ($T_2, S_2$).

Net evaporation over the tropics and net precipitation at high latitudes drive a freshwater flux $F$.
Assuming a linear flow law proportional to density contrast:
$$q = k \Delta \rho = k [\beta (S_1 - S_2) - \alpha (T_1 - T_2)]$$

At steady state with fixed temperature contrast $\Delta T$:
$$\Delta S = \frac{F S_0}{q}$$
Substituting yields a quadratic equation for $q$:
$$q^2 - \sigma q + \gamma F = 0$$
where $\sigma = k \alpha \Delta T$ and $\gamma = k \beta S_0$.


In [ ]:
# Instantiate Stommel Box Model
box_model = StommelBoxModel(Delta_T=20.0, k=1.0)
print(f"Maximum thermal driving (sigma): {box_model.sigma:.2f} Sv")
print(f"Critical freshwater flux (Fc):    {box_model.critical_flux:.4f} Sv")
print(f"Critical overturning (qc):        {box_model.critical_flow:.2f} Sv")

# Plot the stability curve
fig = box_model.plot_stability_curve()
plt.show()


---
## 2. The Stommel–Arons (1960) Abyssal Circulation Model

Stommel and Arons (1960) developed a 2-layer dynamical model for the abyssal ocean.
On an equatorial $\beta$-plane ($f = \beta y$), interior flow in the abyssal layer is in geostrophic balance:
$$u_2 = -\frac{g'}{H \beta y} \frac{\partial \eta}{\partial y}, \quad v_2 = \frac{g'}{H \beta y} \frac{\partial \eta}{\partial x}$$

Substituting into the continuity equation with vertical upwelling $e$:
$$\nabla \cdot (\eta \mathbf{u}_2) = e \implies \frac{g' \eta (H - \eta)}{\beta H y^2} \frac{\partial \eta}{\partial x} = e$$

Integrating westward from the eastern boundary ($x = X_e$, $\eta = h_0$) yields a **cubic equation** for interface height $\eta_i(x, y)$:
$$-\frac{g'}{3} \eta^3 + \frac{g' H}{2} \eta^2 + \left[ \frac{g' h_0^3}{3} - \frac{g' H h_0^2}{2} - e \beta H (X - X_e) y^2 \right] = 0$$

### Key Dynamical Prediction:
Because $e > 0$ and $\beta y > 0$ in the northern hemisphere, interior geostrophic flow $v_2$ is **poleward** (towards the source of deep water!). Mass conservation therefore strictly requires an intense **Deep Western Boundary Current (DWBC)** flowing equatorward.


In [ ]:
sa_model = StommelAronsModel(H=4000.0, h0=2000.0, g_prime=5e-3)
fields_standard = sa_model.compute_fields(so_enhanced=False)

fig = sa_model.plot_solution(fields_standard, title="Stommel-Arons (1960): Standard Uniform Upwelling")
plt.show()


---
## 3. The Straub (1996) Inconsistency

David Straub (Tellus 1996) pointed out a profound contradiction between the two Stommel models:
- Looking at the zonal average $\bar{\eta}(y)$ from the Stommel-Arons solution above, the interface height is **highest in the tropics ($y = 0$)** and slopes downward toward high latitudes!
- If the depth-averaged density of a column is:
  $$\bar{\rho}(y) = \rho_1 \frac{\bar{\eta}}{H} + \rho_2 \left(1 - \frac{\bar{\eta}}{H}\right)$$
  then the ocean column is **denser in the tropics than at high latitudes!**
- Therefore, the basic premise of Stommel's box model ($q = k \Delta \rho$) **contradicts the thermal wind and geostrophic balance** of an unforced, inviscid basin.


---
## 4. Southern Ocean Winds & Gnanadesikan's Pycnocline Model

Toggweiler & Samuels (1995) showed that Southern Ocean winds over the Drake Passage latitude band drive Ekman divergence that pulls deep water to the surface, exerting a strong control on the North Atlantic overturning.

Anand Gnanadesikan (1999) unified this into a conceptual model balancing four transports:
$$q_n = q_{Ek} - q_{ed} + q_u$$
where:
- $q_{Ek} = \frac{\tau L_{gap}}{\rho_0 f}$ (Wind-driven Southern Ocean Ekman transport)
- $q_{ed} = \epsilon D$ with $\epsilon = \frac{A_I L_w}{L_y}$ (Mesoscale eddy return flux)
- $q_u = \frac{K_v A}{D}$ (Low-latitude diapycnal upwelling)
- $q_n = \eta D^2 \Delta \rho$ (Northern sinking)

This allows $D(q_n)$ to vary dynamically and produces the **asymmetric stability curve** derived in the next section.


---
## 5. Stability Curve for the Gnanadesikan Model

The pycnocline depth $D(q_n)$ determined above is the missing ingredient in the Stommel salinity closure. Combining the northern sinking relation
$$q_n = \eta D^2 \Delta\rho$$
with the salinity balance
$$\Delta\rho = \alpha \Theta - \beta S_0 \frac{F}{q_n},$$
gives the equilibrium freshwater flux for the Gnanadesikan model:
$$F(q_n) = \frac{q_n}{\beta S_0}\left[\alpha \Theta - \frac{q_n}{\eta D(q_n)^2}\right].$$

Plotting $q_n$ against $F$ gives the **stability curve for the Gnanadesikan model**:
- The upper branch (blue) is the **stable** overturning regime.
- The lower branch (red dashed) is **unstable**.
- The branches meet at a saddle-node **tipping point**, $F_c \approx 3.1$ Sv and $q_c \approx 27$ Sv for the standard parameters.

The dedicated routine `gnanadesikan_f_vs_overturning.py` draws this curve on its own.


In [ ]:
fig = plot_single_f_vs_overturning()
plt.show()


---
## 6. Can Southern Ocean Winds Reconcile the Models? (Greatbatch & Lu 2003)

Greatbatch & Lu (2003) showed that in a shallow-water model, if the damping time scale of interface anomalies is much shorter than the propagation time scale ($R \ll 1$), Stommel-Arons dynamics govern the flow.

### The Numerical Experiment:
If we impose non-uniform upwelling $e(y)$ in the Stommel-Arons model (enhanced in the Southern Ocean to mimic Drake Passage Ekman divergence) and vary the total circulation $q$:
1. We solve for the altered 2D interface height field $\eta(x, y)$.
2. We compute the resulting interhemispheric density difference $\Delta\bar{\rho} = \bar{\rho}_N - \bar{\rho}_S$.
3. We plot $\Delta\bar{\rho}$ against $q$.


In [ ]:
q_vals, delta_rho = run_wind_experiment(num_cases=19)


---
## 7. Synthesis & Conclusions

1. **The Flow Law is Emergent, Not Causal:**  
   The experiment shows that an almost perfect linear relation between $q$ and $\Delta\bar{\rho}$ appears in a purely dynamical Stommel-Arons model forced by Southern Ocean winds. However, the density contrast is **caused by the circulation**, not driving it.
2. **Implications for Climate Stability:**  
   Because the circulation is governed by Southern Ocean wind stress, mesoscale eddy fluxes, and localized polar convection (rather than a simple basin-scale hydraulic density gradient), the ocean overturning circulation is less susceptible to catastrophic collapse via pure advective instability.
